In [1]:
import pandas as pd
from dotenv import load_dotenv
import os
import psycopg2

In [2]:
load_dotenv()
db_conn_string = os.getenv("DATABASE_CONN_STRING")
db_conn = psycopg2.connect(db_conn_string)

In [3]:
data = pd.read_sql("select * from blackjack_odds", db_conn)

C:\Users\amarl\AppData\Local\Temp\ipykernel_5696\1970751133.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql("select * from blackjack_odds", db_conn)


In [4]:
data.to_csv("blackjack_odds.csv")

In [6]:
data = pd.read_csv("blackjack_odds.csv")

In [7]:
data = data[['player_total', 'dealer_card_up', 'double_ev',
       'hit_ev', 'stand_ev', 'split_ev', 'best_action', 'dealer_hit_soft_17',
       'double_after_split', 'blackjack_pays', 'surrender_allowed']]

Include in new:
- original T,T,1.5 (already corrected)
- original T,F,1.5 (already corrected)
- duplicated T,T,1.2 (already corrected)
- duplicated T,F,1.2 (already corrected)
- original F,T,1.5 (not yet corrected)
- original F,F,1.2 (not yet corrected)
- duplicated F,T,1.2 (not yet corrected)
- duplicated F,F,1.5 (not yet corrected)

In [8]:
tt15 = data[data['dealer_hit_soft_17'] & data['double_after_split'] & (data['blackjack_pays'] == 1.5)].reset_index()

In [9]:
tf15 = data[data['dealer_hit_soft_17'] & (~data['double_after_split']) & (data['blackjack_pays'] == 1.5)].reset_index()

In [10]:
tt12 = data[data['dealer_hit_soft_17'] & data['double_after_split'] & (data['blackjack_pays'] == 1.5)].reset_index()
tt12['blackjack_pays'] = 1.2

In [11]:
tf12 = data[data['dealer_hit_soft_17'] & (~data['double_after_split']) & (data['blackjack_pays'] == 1.5)].reset_index()
tf12['blackjack_pays'] = 1.2

In [12]:
ft15 = data[(~data['dealer_hit_soft_17']) & data['double_after_split'] & (data['blackjack_pays'] == 1.5)].reset_index()

In [13]:
ff12 = data[(~data['dealer_hit_soft_17']) & (~data['double_after_split']) & (data['blackjack_pays'] == 1.2)].reset_index()

In [14]:
ft12 = data[(~data['dealer_hit_soft_17']) & data['double_after_split'] & (data['blackjack_pays'] == 1.5)].reset_index()
ft12['blackjack_pays'] = 1.2

In [15]:
ff15 = data[(~data['dealer_hit_soft_17']) & (~data['double_after_split']) & (data['blackjack_pays'] == 1.2)].reset_index()
ff15['blackjack_pays'] = 1.5

In [16]:
data = pd.concat([tt15,tt12,tf15,tf12,ft15,ft12,ff15,ff12])

In [18]:
data.to_csv("blackjack_odds_v2.csv")